# Исследовательский анализ данных (EDA) — Каталог М.Видео
## Задача: Автоматическое извлечение и нормализация характеристик товаров

Данный ноутбук содержит полный исследовательский анализ (EDA) товарных карточек и атрибутов:
1. **Обзор структуры данных и доступных полей** (Title, Description, Raw Text, Seller Attributes).
2. **Анализ текстовых полей**: распределение длины заголовков и описаний в символах и словах.
3. **Распределение категорий** и специфика каждой товарной группы (смартфоны, ноутбуки, ТВ, пылесосы, холодильники, наушники, powerbank).
4. **Разреженность и заполняемость атрибутов** (Fill Rate vs Sparsity).
5. **Анализ справочников и базы знаний** (`catalog_models_kb.json`, `brands.csv`).
6. **Архитектурные выводы**: обоснование каскадного гибридного пайплайна (Tier 1 FastRules + Tier 2 GLiNER Zero-Shot + Tier 3 LLM Guardrails).

In [1]:
import json
import os
from collections import Counter
from pathlib import Path
import pandas as pd

# Пути к данным
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
GOLD_PATH = DATA_DIR / "gold_dataset.json"
SAMPLE_PATH = DATA_DIR / "sample_cards.json"

print(f"Base dir: {BASE_DIR}")
print(f"Gold dataset exists: {GOLD_PATH.exists()}")

## 1. Загрузка данных и первичный обзор
Загрузим размеченный эталонный набор (Gold Dataset) и демонстрационный пул карточек.

In [2]:
with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold_data = json.load(f)

with open(SAMPLE_PATH, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print(f"Количество карточек в Gold Dataset: {len(gold_data)}")
print(f"Количество карточек в Sample Cards: {len(sample_data)}")

# Пример карточки
sample_item = gold_data[0]
print("\nПример структуры карточки:")
print(json.dumps({
    "id": sample_item.get("id"),
    "category": sample_item.get("category"),
    "title": sample_item.get("title"),
    "expected_attributes": sample_item.get("expected_attributes")
}, ensure_ascii=False, indent=2))

## 2. Анализ текстовых характеристик (Title и Description)
Исследуем объёмы текстов, плотность ключевой информации и распределение длин.

In [3]:
all_cards = gold_data + sample_data

rows = []
for c in all_cards:
    t = (c.get("title") or "").strip()
    d = (c.get("description") or c.get("raw_text") or "").strip()
    rows.append({
        "id": c.get("id") or c.get("external_id"),
        "category": c.get("category"),
        "title_len": len(t),
        "title_words": len(t.split()),
        "desc_len": len(d),
        "desc_words": len(d.split()),
    })

df_texts = pd.DataFrame(rows)
df_texts.describe().round(2)

> **Вывод по текстам:**
> - Заголовки (Title) имеют среднюю длину ~50-80 символов (7-12 слов). В них сконцентрированы ключевые сущности: `бренд`, `модель`, `память`, `диагональ`.
> - Описания (Description) варьируются от 120 до 1200+ символов. В длинных описаниях часто присутствуют сложноподчиненные предложения, блоки комплектации («в комплекте наушники») и упоминания сопутствующих аксессуаров. Это требует строгой фильтрации контекста.

## 3. Распределение категорий товаров в каталоге

In [4]:
cat_counts = Counter(item.get("category") for item in gold_data)
df_cats = pd.DataFrame(cat_counts.items(), columns=["Категория", "Количество карточек"])
df_cats["Доля (%)"] = (df_cats["Количество карточек"] / len(gold_data) * 100).round(1)
df_cats.sort_values(by="Количество карточек", ascending=False).reset_index(drop=True)

## 4. Заполняемость и разреженность атрибутов по категориям
Проанализируем, насколько стабильно встречаются ключевые атрибуты в товарах разных типов.

In [5]:
attr_records = []
for item in gold_data:
    cat = item.get("category")
    expected = item.get("expected_attributes", {})
    for attr_name, val in expected.items():
        attr_records.append({
            "category": cat,
            "attribute": attr_name,
            "has_value": bool(val and str(val).strip()),
        })

df_attrs = pd.DataFrame(attr_records)
attr_summary = df_attrs.groupby(["category", "attribute"]).agg(
    total=("has_value", "count"),
    filled=("has_value", "sum"),
).reset_index()
attr_summary["fill_rate_%"] = (attr_summary["filled"] / attr_summary["total"] * 100).round(1)
attr_summary["sparsity_%"] = (100.0 - attr_summary["fill_rate_%"].astype(float)).round(1)
attr_summary.head(25)

## 5. Анализ базы знаний (Catalog KB) и словарей М.Видео
Для предотвращения галлюцинаций LLM и гарантии воспроизводимости в репозитории хранятся эталонные словари:

In [6]:
kb_path = BASE_DIR / "backend" / "data" / "dictionaries" / "catalog_models_kb.json"
brands_path = BASE_DIR / "backend" / "data" / "dictionaries" / "brands.csv"

with open(kb_path, "r", encoding="utf-8") as f:
    kb_data = json.load(f)

df_brands = pd.read_csv(brands_path)

print(f"Эталонных моделей в Catalog KB: {len(kb_data)}")
print(f"Зарегистрированных брендов в FlashText: {len(df_brands)}")
print("\nПримеры записей из базы моделей М.Видео:")
for k in list(kb_data.keys())[:3]:
    print(f"- {k}: {kb_data[k]}")

## 6. Итоговые архитектурные выводы для продакшн-системы

На основе проведенного анализа сформулированы ключевые принципы архитектуры экстрактора:

1. **Каскадный подход (Tiered Funnel):**
   - **Tier 1 (FastRules + FlashText):** Снимает детерминированные сущности (бренд, цвет, числовые параметры RAM/ROM/батареи) со скоростью `< 1 мс` и `Precision = 1.0`.
   - **Tier 2 (GLiNER Zero-Shot):** Извлекает сложные составные сущности (`model`, `cpu`) без жестких регулярок с помощью модели `urchade/gliner_multi-v2.1`.
   - **Tier 3 (Catalog KB + Strict LLM Guardrail):** Точечно заполняет пропуски в характеристиках известных моделей с жесткой валидацией через `AttributeSanityValidator`.

2. **Строгая защита от утечек атрибутов (Category Whitelist):**
   - Категории имеют изолированные белые списки допустимых полей (например, атрибут `os` или `battery` никогда не появится в телевизоре или холодильнике).

3. **Human-in-the-Loop (HITL) калибровка:**
   - При наличии пропусков или низкой средней уверенности (`< 0.87`) карточка автоматически помечается флагом `needs_review: true` и направляется в очередь ручной верификации.